In [ ]:
# Make this notebook work from either fine-tuning/ or fine-tuning/pre-demo/
# (idempotent: re-running is safe)
import os
from pathlib import Path
_here = Path.cwd()
if _here.name in ('pre-demo', 'live-demo'):
    os.chdir(_here.parent)
print('cwd:', Path.cwd())


# Lab 12 · Production Deployment & Lifecycle (MLOps/CI-CD)

A model that works in a notebook is not in production. This lab covers the lifecycle that makes it safe: **list what's deployed**, choose **PTU vs pay-as-you-go**, ship a new version with **blue/green** behind a stable alias, smoke-test it, and **roll back in one line** if it regresses. *Ship daily without fear.*


---
## Step 0 — Enable Foundry tracing

*Wire OpenTelemetry to Application Insights so every model call below shows up live in the Microsoft Foundry portal under **your project → Tracing**.*


In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve()))
from _tracing import enable_foundry_tracing

enable_foundry_tracing(service_name='sutter-live-demo')


---
## Step 1 — What's deployed right now? (LIVE)

Read the live deployments and their SKU/tier so you know your blast radius before changing anything.


In [ ]:
from _advisor import load_catalog_live

models, meta = load_catalog_live()
print(f"{'deployment':30} {'tier':10} {'sku':16} {'live'}")
print('-' * 64)
for m in models:
    print(f"{m['deployment']:30} {str(m.get('tier')):10} "
          f"{str(m.get('sku','-')):16} {m.get('live', meta.get('live'))}")
print(f"\nCatalog source: {meta.get('source')} (live={meta.get('live')})")


---
## Step 2 — Blue/green promotion behind a stable alias

Production code should call a **stable alias**, never a raw deployment name. To ship a new model you point the alias at the new (green) deployment *after* it passes a smoke test — and rollback is just re-pointing the alias. We exercise this live against two real deployments.


In [ ]:
from _advisor import try_build_client

client = try_build_client()

# Stable alias your app imports; swapping its value is the deploy.
ALIAS = {'sutter-prod': 'sutter-sft-deployment'}  # BLUE (current prod)
GREEN = 'sutter-dpo-deployment'                    # candidate next version

def ask(deployment, prompt):
    if client is None:
        return '[mock] ' + prompt[:40]
    r = client.chat.completions.create(model=deployment, max_tokens=60,
        messages=[{'role': 'user', 'content': prompt}])
    return r.choices[0].message.content.strip()

SMOKE = 'In one sentence, how do I refill a prescription with Sutter?'

print('BLUE  (', ALIAS['sutter-prod'], '):', ask(ALIAS['sutter-prod'], SMOKE)[:120])
green_answer = ask(GREEN, SMOKE)
print('GREEN (', GREEN, '):', green_answer[:120])

smoke_ok = bool(green_answer) and 'refill' in green_answer.lower()
if smoke_ok:
    ALIAS['sutter-prod'] = GREEN      # PROMOTE
    print('\n✅ Smoke test passed -> promoted prod to', ALIAS['sutter-prod'])
else:
    print('\n⚠️ Smoke test failed -> kept prod on', ALIAS['sutter-prod'])


---
## Step 3 — One-line rollback

If post-deploy metrics dip, revert instantly. No redeploy, no downtime.


In [ ]:
PREVIOUS = 'sutter-sft-deployment'
ALIAS['sutter-prod'] = PREVIOUS   # rollback
print('↩️  Rolled back: sutter-prod ->', ALIAS['sutter-prod'])


---
## Step 4 — Replace-in-production: PTU + CI/CD

Two production decisions the alias pattern enables:

- **PTU vs pay-as-you-go** — `GlobalStandard` (pay-go) is great for spiky/dev traffic; **Provisioned Throughput Units (PTU)** give reserved, predictable latency + price for steady Member Services load.
- **CI/CD** — create the green deployment, run evals (Lab 07/13), then swap the alias from a pipeline.


In [ ]:
CICD_REFERENCE = '''
# --- replace-in-production: deploy + smoke + promote (Azure CLI) ---
# 1) create GREEN deployment (PTU example)
az cognitiveservices account deployment create \\
  -g $RG -n $ACCT --deployment-name sutter-green \\
  --model-name gpt-4o --model-version 2024-08-06 \\
  --sku-name ProvisionedManaged --sku-capacity 50
# 2) run eval gate (Lab 07/13) -> must pass
# 3) promote: point your app config alias 'sutter-prod' at sutter-green
# 4) rollback: point alias back at the previous deployment
'''
print(CICD_REFERENCE)


---
## Takeaways

- Apps call a **stable alias**, never a raw deployment — that single indirection makes blue/green + rollback trivial.
- **PTU** buys predictable latency/price for steady load; **pay-go** wins for spiky/dev.
- Promotion is **gated by evals**, not vibes (wire Lab 07/13 into CI).
- Rollback is **one line** and zero downtime.

*← The Decision Advisor (Lab 09) routes the `not_production_ready` flag here.*
